## 🏗️ Mini Project — Random Forest Regression

#### Customer Lifetime Value (CLV) Predictor

In [34]:
import numpy as np
import pandas as pd

np.random.seed(21)
n_customers = 400

# Relevant features
avg_order_value = np.random.uniform(20, 200, n_customers)
purchase_frequency = np.random.poisson(5, n_customers)  # per year
tenure_months = np.random.uniform(1, 60, n_customers)
support_tickets = np.random.poisson(2, n_customers)

# Irrelevant/noise features (real-world data always has some!)
random_noise_1 = np.random.randn(n_customers)
random_noise_2 = np.random.randn(n_customers)
signup_day_of_week = np.random.randint(0, 7, n_customers)

X = np.column_stack([avg_order_value, purchase_frequency, tenure_months, 
                       support_tickets, random_noise_1, random_noise_2, signup_day_of_week])

# True relationship (complex, non-linear interactions)
CLV = (avg_order_value * purchase_frequency * 0.5 
       + tenure_months * 8 
       - support_tickets * 15 
       + np.random.normal(0, 50, n_customers))
CLV = np.maximum(CLV, 0)  # CLV can't be negative

y = CLV


#### Task to perform
1. Train/test split (80/20)
2. Baseline comparison: single Decision Tree vs Random Forest across 5 different seeds (compare both mean and standard deviation)
3. Tune max_features: try 2-3 values (e.g. 2, 4, 7/all) and find the best (hint: only 4 features are genuinely relevant here)
4. Tune n_estimators: try 50, 100, 200 and see where performance plateaus (more trees stop helping: "diminishing returns")
5. Feature importance: use scikit-learn's built-in feature importances

In [ ]:
from sklearn. model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn. metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

tree_results = []
rf_results = []

for seed in [22, 42, 62]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=seed)

    for depth in [2, 4, 6]:
        tree = DecisionTreeRegressor(max_depth= depth, random_state= seed)
        tree.fit(X_train, y_train)
        tree_pred = tree.predict(X_test)

        tree_mse = mean_squared_error(y_test, tree_pred)

        tree_results.append(
            {"Seed": seed, "Max Depth": depth, "MSE": round(tree_mse, 2)}
        )

    for estimator in [50, 100, 200]:
        for feature in [2, 4, 7]:
            for depth in [2, 4, 6]:

                rf = RandomForestRegressor(n_estimators= estimator, max_depth= depth, max_features= feature, random_state= seed)
                rf.fit(X_train, y_train)
                rf_pred = rf.predict(X_test)

                rf_mse = mean_squared_error(y_test, rf_pred)

                rf_results.append(
                    {
                        "Seed": seed,
                        "Estimators": estimator,
                        "Max Features": feature,
                        "Max Depth": depth,
                        "MSE": round(rf_mse, 2),
                    }
                )

df_tree = pd.DataFrame(tree_results)
df_rf = pd.DataFrame(rf_results)



In [36]:
top5_tree = df_tree.sort_values(by= 'MSE', ascending=True).head(5)
top5_tree

,Seed,Max Depth,MSE
2,22,6,9996.82
5,42,6,10739.21
1,22,4,13021.51
4,42,4,13652.59
8,62,6,14401.38


In [37]:
top5_rf = df_rf.sort_values(by = "MSE", ascending= True).head(5)
top5_rf

,Seed,Estimators,Max Features,Max Depth,MSE
26,22,200,7,6,5766.78
17,22,100,7,6,5925.87
53,42,200,7,6,6014.78
80,62,200,7,6,6015.47
44,42,100,7,6,6082.09


From the above i select seed = 22, Estimator = 200, Max Features = 7 and Max Depth = 6

In [38]:
importances = rf.feature_importances_
for i, imp in enumerate(importances):
       print(f"Feature {i}: {imp:.3f}")

Feature 0: 0.415
Feature 1: 0.254
Feature 2: 0.284
Feature 3: 0.007
Feature 4: 0.012
Feature 5: 0.021
Feature 6: 0.007


In [39]:
from sklearn. model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn. metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

tree_error = []
rf_error = []

for seed in [22, 42, 62]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= seed)


    tree = DecisionTreeRegressor(max_depth= 6, random_state= seed)
    tree.fit(X_train, y_train)
    tree_pred = tree.predict(X_test)

    tree_mse = mean_squared_error(y_test, tree_pred)
    tree_error.append(tree_mse)

    rf = RandomForestRegressor(n_estimators= 200, max_depth= 6, max_features= 7, random_state=seed)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    rf_mse = mean_squared_error(y_test, rf_pred)

    rf_error.append(rf_mse)

    print(f'Seed {seed}: Tree MSE {tree_mse:.2f}, Forest MSE {rf_mse:.2f}')


print(f"\nTree  -> Mean: {np.mean(tree_error):.2f}, Std: {np.std(tree_error):.2f}")
print(f"Forest -> Mean: {np.mean(rf_error):.2f}, Std: {np.std(rf_error):.2f}")


Seed 22: Tree MSE 9996.82, Forest MSE 5766.78
Seed 42: Tree MSE 10739.21, Forest MSE 6014.78
Seed 62: Tree MSE 14401.38, Forest MSE 6015.47

Tree  -> Mean: 11712.47, Std: 1925.35
Forest -> Mean: 5932.34, Std: 117.07


In [40]:
# Build a new model using the best parameters explicitly
best_seed = 22
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=best_seed)

final_rf = RandomForestRegressor(n_estimators=200, max_depth=6, max_features=7, random_state=best_seed)
final_rf.fit(X_train, y_train)

importances = final_rf.feature_importances_
for i, imp in enumerate(importances):
    print(f"Feature {i}: {imp:.3f}")

Feature 0: 0.433
Feature 1: 0.242
Feature 2: 0.278
Feature 3: 0.009
Feature 4: 0.011
Feature 5: 0.021
Feature 6: 0.006
